# 06. Prefix XOR, Prefix Product and 2D Difference Array

The prefix idea works for **any operation that can be undone**: sum (subtract), XOR (XOR again), product (divide, with care for zeros), counts. And it extends to **2D**.

**Learning order:** PREFIX XOR → COUNT XOR SUBARRAYS → PREFIX PRODUCT → SPLIT ARRAY → MIN START VALUE → 2D DIFFERENCE ARRAY → INTERVIEW THINKING

## 1. Prefix XOR: range XOR in O(1)

Because `a ^ a = 0`, `prefix[r + 1] ^ prefix[l]` cancels everything before `l`: it equals the XOR of `arr[l..r]`.

In [1]:
def xor_queries(arr, queries):
    prefix = [0]
    for x in arr:
        prefix.append(prefix[-1] ^ x)
    return [prefix[r + 1] ^ prefix[l] for l, r in queries]

print(xor_queries([1, 3, 4, 8], [[0, 1], [1, 2], [0, 3], [3, 3]]))   # [2, 7, 14, 8]

[2, 7, 14, 8]


## 2. Count subarrays with XOR equal to K

Same trick as "subarray sum equals k": `prefix_xor_j ^ prefix_xor_i = k` ⇔ `prefix_xor_i = prefix_xor_j ^ k`. Look up `current ^ k` in a hash map of counts (start with `{0: 1}`).

In [2]:
from collections import defaultdict

def count_xor_subarrays(nums, k):
    seen = defaultdict(int)
    seen[0] = 1
    cur = count = 0
    for x in nums:
        cur ^= x
        count += seen[cur ^ k]
        seen[cur] += 1
    return count

print(count_xor_subarrays([4, 2, 2, 6, 4], 6))   # 4

4


## 3. Product of the Last K Numbers (prefix product with zeros)

Keep **prefix products**. Product of the last `k` = `prefix[-1] / prefix[-1-k]`. A **zero** breaks division: when a `0` arrives, **reset** the list (every window containing the zero has product 0). If `k` is bigger than the list since the last zero, the answer is `0`.

In [3]:
class ProductOfNumbers:
    def __init__(self):
        self.prefix = [1]

    def add(self, num):
        if num == 0:
            self.prefix = [1]                      # everything before is multiplied by 0
        else:
            self.prefix.append(self.prefix[-1] * num)

    def get_product(self, k):
        if k >= len(self.prefix):
            return 0                               # the window includes a zero
        return self.prefix[-1] // self.prefix[-1 - k]

p = ProductOfNumbers()
for x in [3, 0, 2, 5, 4]:
    p.add(x)
print(p.get_product(2), p.get_product(3), p.get_product(4))   # 20 40 0
p.add(8)
print(p.get_product(2))   # 32

20 40 0
32


## 4. Number of Ways to Split Array

Count split positions `i` where `sum(left) >= sum(right)`. One pass with a running `left` and `total - left`.

In [4]:
def ways_to_split(nums):
    total, left, ways = sum(nums), 0, 0
    for i in range(len(nums) - 1):
        left += nums[i]
        if left >= total - left:
            ways += 1
    return ways

print(ways_to_split([10, 4, -8, 7]), ways_to_split([2, 3, 1, 0]))   # 2 2

2 2


## 5. Minimum Value to Get Positive Step by Step Sum

The running sum must **never drop below 1**. Find the **lowest prefix sum** `m`; the start value must be `1 - m` (at least 1).

In [5]:
def min_start_value(nums):
    cur = lowest = 0
    for x in nums:
        cur += x
        lowest = min(lowest, cur)
    return 1 - lowest

print(min_start_value([-3, 2, -3, 4, 2]), min_start_value([1, 2]))   # 5 1

5 1


## 6. 2D Difference Array: add a value to a rectangle

In 1D: `diff[l] += x`, `diff[r+1] -= x`. In 2D, adding `v` to the rectangle `(r1, c1)` to `(r2, c2)` changes **four** corners:

```
diff[r1][c1]         += v
diff[r1][c2 + 1]     -= v
diff[r2 + 1][c1]     -= v
diff[r2 + 1][c2 + 1] += v
```

Then take a **2D prefix sum** of `diff` to get the final grid. Many rectangle updates cost O(1) each; one O(m·n) pass at the end.

In [6]:
def apply_rectangles(rows, cols, updates):                # updates: (r1, c1, r2, c2, v)
    diff = [[0] * (cols + 1) for _ in range(rows + 1)]
    for r1, c1, r2, c2, v in updates:
        diff[r1][c1] += v
        diff[r1][c2 + 1] -= v
        diff[r2 + 1][c1] -= v
        diff[r2 + 1][c2 + 1] += v
    grid = [[0] * cols for _ in range(rows)]
    for r in range(rows):
        for c in range(cols):
            up = grid[r - 1][c] if r else 0
            left = grid[r][c - 1] if c else 0
            diag = grid[r - 1][c - 1] if r and c else 0
            grid[r][c] = diff[r][c] + up + left - diag      # 2D prefix sum of diff
    return grid

print(apply_rectangles(3, 3, [(0, 0, 1, 1, 1), (1, 1, 2, 2, 2)]))   # [[1, 1, 0], [1, 3, 2], [0, 2, 2]]

[[1, 1, 0], [1, 3, 2], [0, 2, 2]]


## 7. Interview thinking

**Signals:** "XOR of a range", "subarray XOR equals k", "product of the last k", "many rectangle updates", "the running sum must stay ≥ 1".

**Say:** "XOR is its own inverse, so prefix XOR works exactly like prefix sums. For products I reset at zero because division by zero is impossible."

**Common mistakes:** forgetting `{0: 1}` in the XOR map, dividing by a prefix containing zero, wrong sign pattern in the 2D difference (`+ - - +`), off-by-one in `r2 + 1` / `c2 + 1` (allocate `rows + 1`, `cols + 1`).

## Quick Revision

- Prefix works for invertible operations: sum, XOR, count, product (reset on zero).
- XOR: `prefix[r+1] ^ prefix[l]`; subarray XOR k → map of `current ^ k`.
- Min start value: `1 - lowest prefix`.
- 2D diff: `+ - - +` at the four corners, then 2D prefix sum.